In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# Cấu hình sys.path bằng đường dẫn tương đối (chạy tại thư mục hiện tại)
sys.path = [p for p in sys.path if p is not None]
src_path = os.getcwd() 
if src_path not in sys.path:
    sys.path.append(src_path)

# Import các module dùng chung từ folder 'src' do nhóm xây dựng
from src.data_pipeline import load_raw_data, handle_missing_values
from src.feature_engineering import apply_log_transform, remove_high_correlation, select_top_features
from src.internal_tracker import ExperimentTracker

# Import bộ 4 mô hình học máy Tree-based mạnh nhất (A-Team)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# Khởi tạo tracker lưu log cho phiên bản v5
tracker = ExperimentTracker(project_root=src_path, version="v5")
print("✅ BƯỚC 1 HOÀN THÀNH: Đã kết nối module 'src' và bộ 4 mô hình học máy thành công!")

✅ BƯỚC 1 HOÀN THÀNH: Đã kết nối module 'src' và bộ 4 mô hình học máy thành công!


In [2]:
# Chỉ định chính xác vị trí thư mục 'data' nằm cùng cấp với file notebook
data_dir = os.path.join(src_path, 'data')

# Gọi hàm load dữ liệu chuẩn của nhóm
train_df, test_df, _ = load_raw_data(data_dir)

target = 'Class'
X_raw = train_df.drop(columns=['Id', target, 'Artist Name', 'Track Name'], errors='ignore')
y = train_df[target]
X_test_raw = test_df.drop(columns=['Id', 'Artist Name', 'Track Name'], errors='ignore')

# Thực hiện chuỗi tiền xử lý dữ liệu (Data Pipeline)
X_filled = handle_missing_values(X_raw)
X_test_filled = handle_missing_values(X_test_raw)

log_cols = ['speechiness', 'acousticness', 'instrumentalness']
X_log = apply_log_transform(X_filled, columns=log_cols)
X_test_log = apply_log_transform(X_test_filled, columns=log_cols)

X_nocorr, dropped_features = remove_high_correlation(X_log, threshold=0.90)
X_test_nocorr = X_test_log.drop(columns=dropped_features, errors='ignore')

X_selected, selector_model = select_top_features(X_nocorr, y, k=10)
X_test_selected = selector_model.transform(X_test_nocorr)

print(f"✅ BƯỚC 2 HOÀN THÀNH: Dữ liệu đã xử lý xong. Kích thước tập Train mới: {X_selected.shape}")

✅ BƯỚC 2 HOÀN THÀNH: Dữ liệu đã xử lý xong. Kích thước tập Train mới: (14396, 10)


In [3]:
# Cấu hình StratifiedKFold chia dữ liệu thành 5 phần bằng nhau giữ nguyên tỷ lệ class
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Chuẩn bị ma trận trống để hứng xác suất dự đoán Out-Of-Fold (OOF) phục vụ đánh giá và làm V6
oof_lgb = np.zeros((len(X_selected), 11))
oof_xgb = np.zeros((len(X_selected), 11))
oof_cat = np.zeros((len(X_selected), 11))
oof_rf  = np.zeros((len(X_selected), 11))

print("⏳ Đang tiến hành chạy thử nghiệm song song 4 mô hình, vui lòng đợi...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_selected, y)):
    X_train, y_train = X_selected[train_idx], y.iloc[train_idx]
    X_val, y_val = X_selected[val_idx], y.iloc[val_idx]
    
    # 1. Huấn luyện LightGBM Classifier (Trọng số Balanced)
    model_lgb = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300, verbose=-1)
    model_lgb.fit(X_train, y_train)
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)
    
    # 2. Huấn luyện XGBoost Classifier (Tính toán sample_weight theo fold dữ liệu)
    sw_xgb = compute_sample_weight(class_weight='balanced', y=y_train)
    model_xgb = XGBClassifier(random_state=42, n_estimators=300, eval_metric='mlogloss')
    model_xgb.fit(X_train, y_train, sample_weight=sw_xgb)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)
    
    # 3. Huấn luyện CatBoost Classifier (Trọng số tự động Balanced)
    model_cat = CatBoostClassifier(auto_class_weights='Balanced', random_state=42, iterations=300, verbose=0)
    model_cat.fit(X_train, y_train)
    oof_cat[val_idx] = model_cat.predict_proba(X_val)
    
    # 4. Huấn luyện RandomForest Classifier (Trọng số Balanced)
    model_rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200)
    model_rf.fit(X_train, y_train)
    oof_rf[val_idx] = model_rf.predict_proba(X_val)
    
    print(f"   ➔ Đang chạy thử nghiệm hệ thống: Fold {fold + 1}/5 hoàn tất.")

print("✅ BƯỚC 3 HOÀN THÀNH: Đã huấn luyện thử nghiệm xong cả 4 mô hình cốt lõi!")

⏳ Đang tiến hành chạy thử nghiệm song song 4 mô hình, vui lòng đợi...


c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ➔ Đang chạy thử nghiệm hệ thống: Fold 1/5 hoàn tất.


c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ➔ Đang chạy thử nghiệm hệ thống: Fold 2/5 hoàn tất.


...

   ➔ Đang chạy thử nghiệm hệ thống: Fold 3/5 hoàn tất.


c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ➔ Đang chạy thử nghiệm hệ thống: Fold 4/5 hoàn tất.


c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ➔ Đang chạy thử nghiệm hệ thống: Fold 5/5 hoàn tất.
✅ BƯỚC 3 HOÀN THÀNH: Đã huấn luyện thử nghiệm xong cả 4 mô hình cốt lõi!


In [4]:
# Kết hợp xác suất (Soft Voting) có trọng số ưu tiên các thuật toán Boosting cây mạnh
w_lgb, w_xgb, w_cat, w_rf = 0.35, 0.35, 0.20, 0.10
blend_oof_probs = (w_lgb * oof_lgb) + (w_xgb * oof_xgb) + (w_cat * oof_cat) + (w_rf * oof_rf)
final_oof_preds = np.argmax(blend_oof_probs, axis=1)

# Tính toán điểm số tổng hợp
v5_macro_f1 = f1_score(y, final_oof_preds, average='macro')

print("\n================ BÁO CÁO ĐÁNH GIÁ HIỆU NĂNG V5 ================")
print(f"🎯 Điểm số F1 Macro thu được trên tập dữ liệu kiểm thử OOF: {v5_macro_f1:.5f}")
print("==============================================================")
print(classification_report(y, final_oof_preds))

# Thực hiện lưu thông tin, danh sách đặc trưng và xuất file báo cáo log cho nhóm
tracker.log_artifact("feature_list_v5.txt", list(X_nocorr.columns[selector_model.get_support()]))
tracker.log_metrics({"v5_final_macro_f1": v5_macro_f1})
tracker.log_text_report("class_weight_analysis.txt", classification_report(y, final_oof_preds))

print("✅ BƯỚC 4 HOÀN THÀNH: Đã tính toán đánh giá thành công và ghi nhận báo cáo vào 'class_weight_analysis.txt'!")


================ BÁO CÁO ĐÁNH GIÁ HIỆU NĂNG V5 ================
🎯 Điểm số F1 Macro thu được trên tập dữ liệu kiểm thử OOF: 0.56019
              precision    recall  f1-score   support

           0       0.65      0.73      0.69       500
           1       0.08      0.07      0.07      1098
           2       0.46      0.49      0.48      1018
           3       0.78      0.72      0.75       322
           4       0.62      0.69      0.65       310
           5       0.67      0.71      0.69      1157
           6       0.32      0.31      0.31      2069
           7       0.93      0.93      0.93       461
           8       0.58      0.60      0.59      1483
           9       0.51      0.51      0.51      2019
          10       0.48      0.49      0.49      3959

    accuracy                           0.50     14396
   macro avg       0.55      0.57      0.56     14396
weighted avg       0.49      0.50      0.49     14396

✅ BƯỚC 4 HOÀN THÀNH: Đã tính toán đánh giá thành công v

In [5]:
print("⏳ Đang chạy mô hình triển khai trên toàn bộ tập dữ liệu để dự đoán tập Test...")
# Sử dụng mô hình đại diện tối ưu huấn luyện trên toàn bộ tập dữ liệu đã scale
deploy_model = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300, verbose=-1)
deploy_model.fit(X_selected, y)

# Dự đoán nhãn cho tập Test
test_predictions = deploy_model.predict(X_test_selected)

# Đóng gói cấu trúc DataFrame nộp bài theo chuẩn quy định
submission_v5 = pd.DataFrame({
    'Id': test_df['Id'],
    'Class': test_predictions
})

# Xuất thành tập tin kết quả nộp bài lưu ngay tại thư mục hiện tại của file notebook
submission_v5.to_csv('submission_version5.csv', index=False)
print("🎉 THÀNH CÔNG! Đã tạo và xuất file kết quả nộp bài: 'submission_version5.csv'")

# Lưu trữ lại toàn bộ mảng dữ liệu OOF npy làm tài nguyên đầu vào cho bước Stacking của V6
np.save('oof_lgb_v5.npy', oof_lgb)
np.save('oof_xgb_v5.npy', oof_xgb)
np.save('oof_cat_v5.npy', oof_cat)
np.save('oof_rf_v5.npy', oof_rf)
print("💾 Đã lưu trữ toàn bộ các file mảng dự đoán .npy cục bộ thành công!")
print("\n🏁 QUY TRÌNH HỦY DIỆT VERSION 5 HOÀN THÀNH ĐẦY ĐỦ 100% CÁC BƯỚC CHUẨN!")

⏳ Đang chạy mô hình triển khai trên toàn bộ tập dữ liệu để dự đoán tập Test...
🎉 THÀNH CÔNG! Đã tạo và xuất file kết quả nộp bài: 'submission_version5.csv'
💾 Đã lưu trữ toàn bộ các file mảng dự đoán .npy cục bộ thành công!

🏁 QUY TRÌNH HỦY DIỆT VERSION 5 HOÀN THÀNH ĐẦY ĐỦ 100% CÁC BƯỚC CHUẨN!


c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
